# Meta-Controller: Selecting the Right Agentic Architecture per Task

Every pattern built so far in this repo — direct LLM calls, ReAct tool-use loops, multi-step planning, debate-style deliberation — is *good at some tasks and wasteful or wrong for others*. A single fixed-answer LLM call is perfect for "what is the capital of France?" and useless for "should we adopt a 4-day work week?". A multi-step planner is overkill for a one-fact lookup, and a ReAct search loop is pointless when the answer never left the model's training data.

Most systems pick one architecture up front and force every query through it. This notebook builds a **Meta-Controller**: a router that sits *above* all of these architectures and, for each incoming task, decides **which whole strategy** should handle it — then dispatches to that strategy's implementation.

### Definition

A **Meta-Controller** is a classification-and-dispatch layer whose decision variable is *which agentic architecture to run*, not which specialist, tool, or prompt variant to use inside one fixed architecture. Given a task, it reasons about the *shape* of the task (Is it a simple lookup? Does it need live/external information? Does it decompose into sub-parts? Is it a matter of opinion or trade-offs?) and picks the one strategy best suited to that shape, along with a stated reason for the choice.

### High-level Workflow

1. **Classify:** The Meta-Controller LLM call receives the raw task and a description of each available strategy, and returns a structured decision: `strategy` (one of a fixed set) + `reasoning` (why that strategy fits this task).
2. **Dispatch:** The controller looks up the callable registered for the chosen strategy and invokes it with the task.
3. **Execute:** The chosen strategy runs *end-to-end on its own terms* — a single call, a tool loop, a plan/execute/synthesize sequence, or a two-stance debate — each with its own internal control flow, invisible to the controller.
4. **Return:** The strategy's output is handed back, tagged with which strategy produced it and why, so the routing decision stays inspectable.

### When to Use / Applications
* **General-purpose assistants** that must handle everything from trivia to research to open-ended judgment calls without hardcoding a single workflow.
* **Cost/latency-sensitive systems** where running the expensive multi-step or multi-agent architectures on every query (including trivial ones) would be wasteful.
* **Platforms that keep growing new agentic patterns** — new strategies can be added by registering one more callable and one more line in the classifier's prompt, without touching the others.

### Strengths & Weaknesses
* **Strengths:**
  * **Right-sized cost:** simple questions get a single cheap call; only genuinely complex or live-data tasks pay for the expensive architectures.
  * **Composability:** each strategy is a self-contained, independently testable unit; the controller doesn't need to know *how* a strategy works internally, only *when* to pick it.
  * **Inspectable decisions:** the structured `reasoning` field makes every routing choice auditable, which matters when the "wrong" architecture would silently produce a plausible-looking but poor answer (e.g. a single-shot guess at a live-data question).
* **Weaknesses:**
  * **Classification errors cascade:** if the Meta-Controller misjudges the task shape (e.g. treats a live-data question as a simple fact), the chosen strategy cannot recover — there's no fallback in this minimal design.
  * **Coarse granularity:** picking one strategy per whole task can't handle tasks that are a genuine *mixture* (e.g. "look up X, then debate whether it's a good idea") without a smarter compositional controller.
  * **Extra latency for the classification step itself:** one LLM call is spent purely on deciding, before any real work starts.

### How This Differs from Ordinary Routing and from a Blackboard Controller

It is easy to conflate this with two other patterns already covered in this repo — the distinction is the entire point of this notebook:

* **Ordinary routing (`Workflow_and_Agent_Patterns/06_Router`)** picks between a small set of *prompt variants, tools, or specialist branches inside one fixed workflow graph*. The overall shape of execution — "classify, then run exactly one branch, then return" — never changes; only *which branch* changes. It answers "which specialist/tool handles this?"
* **A Blackboard's Controller** *sequences* a fixed roster of specialists that all read and write to one shared blackboard *within one architecture*. It decides "who acts next, given the current state of the board?" — but the architecture itself (blackboard + specialists + controller loop) is a single, unchanging structure applied to every task.
* **A Meta-Controller** operates one level above both of these. Its decision is "which *entire architecture* — direct answer, ReAct tool loop, multi-step planner, debate — should even be constructed for this task?" It does not sequence steps or specialists *within* an architecture; it chooses *among architectures*, each of which may have a completely different internal control flow (some loop, some don't; some use tools, some don't; some involve multiple LLM personas, some use just one).

In short: routing and blackboard-controller decisions happen *inside* a fixed architecture; a Meta-Controller decision happens *before* any architecture is even chosen.

## Phase 0: Foundation & Setup

We use this repo's shared `helpers.get_llm()` factory (platform-aware: Groq on Windows, Databricks on macOS) instead of instantiating a provider client directly. For the live-info strategy we use `langchain-tavily`'s `TavilySearch`, matching the convention used elsewhere in this repo.

### Step 0.1: Installing Core Libraries

**What we are going to do:**
Install the libraries this notebook needs: `langchain-tavily` for the ReAct strategy's search tool, `pydantic` for structured output schemas, and `rich` for readable console output.

In [ ]:
# !pip install -q -U langchain langchain-tavily pydantic rich python-dotenv


### Step 0.2: Importing Libraries and Initializing the LLM

**What we are going to do:**
Import what we need, load `.env` (this repo's convention keeps `TAVILY_API_KEY` etc. at the project root), and initialize the LLM through the shared factory rather than a provider-specific class.

In [ ]:
# ============ IMPORTS & ENVIRONMENT ============
import os
from enum import Enum
from typing import List, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from rich.console import Console
from rich.markdown import Markdown

from langchain_core.tools import tool
from langchain_tavily import TavilySearch

from helpers import get_llm

load_dotenv()

if not os.environ.get("TAVILY_API_KEY"):
    print("TAVILY_API_KEY not found. Please set it in a .env file at the project root.")

console = Console()
llm = get_llm()
print("LLM initialized.")


## Phase 1: The Strategy Implementations

**What we are going to do:**
Define four lightweight, self-contained "strategies" — each a plain callable `(query: str) -> str`. These are simplified stand-ins for the fuller architectures built elsewhere in this repo (e.g. ReAct in `05_AI_Agent_Fundamentals`, planning in `Workflow_and_Agent_Patterns/02_Planning`). The point of this notebook is the **selection logic** that picks between them, not re-implementing each architecture in full:

1. **`run_direct_answer`** — a single LLM call. Best for simple, self-contained factual questions the model already knows.
2. **`run_react_tool_use`** — a small ReAct-style loop: the LLM decides whether it needs to search, calls a `web_search` tool if so, then answers. Best for questions needing live or external information.
3. **`run_multi_step_planning`** — decomposes the task into sub-steps, answers each, then synthesizes. Best for multi-part questions that bundle several distinct sub-questions together.
4. **`run_debate`** — generates a "for" stance and an "against" stance, then a judge synthesizes a balanced verdict. Best for opinion / trade-off questions that don't have one factual answer.

In [ ]:
# ============ STRATEGY 1: DIRECT ANSWER ============
def run_direct_answer(query: str) -> str:
    """Single LLM call — for simple, self-contained factual questions."""
    console.print("--- STRATEGY [direct_answer]: Answering in a single call... ---")
    response = llm.invoke(
        f"Answer the following question directly and concisely.\n\nQuestion: {query}"
    )
    return response.content


In [ ]:
# ============ STRATEGY 2: REACT TOOL USE ============
tavily_search_tool = TavilySearch(max_results=3)


@tool
def web_search(search_query: str) -> str:
    """Search the web for up-to-date or external information."""
    console.print(f"    [tool] web_search('{search_query}')")
    return str(tavily_search_tool.invoke(search_query))


class ReactDecision(BaseModel):
    """Whether another search is needed, or the loop can answer now."""
    need_search: bool = Field(description="True if a web search is still needed before answering.")
    search_query: Optional[str] = Field(
        default=None, description="The query to search for, if need_search is True."
    )


def run_react_tool_use(query: str, max_turns: int = 3) -> str:
    """A small ReAct loop: reason -> (optionally) search -> repeat -> answer.

    For questions that need live or external information the model cannot
    reliably know on its own.
    """
    console.print("--- STRATEGY [react_tool_use]: Running ReAct loop... ---")
    observations: List[str] = []
    decision_llm = llm.with_structured_output(ReactDecision)

    for turn in range(max_turns):
        context = "\n".join(observations) if observations else "(no searches performed yet)"
        decision = decision_llm.invoke(
            f"""Question: {query}

Observations so far:
{context}

Decide whether another web search is needed to answer the question, or whether
enough information has already been gathered to answer confidently."""
        )
        if not decision.need_search:
            break
        result = web_search.invoke(decision.search_query or query)
        observations.append(f"Search('{decision.search_query}') -> {result}")

    context = "\n".join(observations) if observations else "(no searches performed)"
    final = llm.invoke(
        f"""Using the observations below, answer the question concisely.

Question: {query}

Observations:
{context}"""
    )
    return final.content


In [ ]:
# ============ STRATEGY 3: MULTI-STEP PLANNING ============
class SubSteps(BaseModel):
    """An ordered list of sub-questions that together cover the full task."""
    steps: List[str] = Field(description="Ordered list of self-contained sub-questions.")


def run_multi_step_planning(query: str) -> str:
    """Decompose -> answer each sub-step -> synthesize.

    For multi-part questions that bundle several distinct sub-questions together.
    """
    console.print("--- STRATEGY [multi_step_planning]: Decomposing into sub-steps... ---")
    planner_llm = llm.with_structured_output(SubSteps)
    plan = planner_llm.invoke(
        f"Break the following task into an ordered list of self-contained sub-questions "
        f"that together fully answer it.\n\nTask: {query}"
    )
    console.print(f"    [plan] {plan.steps}")

    answers = []
    for step in plan.steps:
        answer = llm.invoke(f"Answer this sub-question concisely: {step}").content
        console.print(f"    [sub-step] {step} -> {answer[:80]}...")
        answers.append(f"Sub-question: {step}\nAnswer: {answer}")

    context = "\n\n".join(answers)
    final = llm.invoke(
        f"""Combine the sub-answers below into one coherent final answer to the original task.

Original task: {query}

Sub-answers:
{context}"""
    )
    return final.content


In [ ]:
# ============ STRATEGY 4: DEBATE ============
def run_debate(query: str) -> str:
    """Two opposing stances, then a judge synthesizes a balanced verdict.

    For opinion / trade-off questions that don't have one factual answer.
    """
    console.print("--- STRATEGY [debate]: Generating opposing stances... ---")
    for_case = llm.invoke(
        f"Argue IN FAVOR of the following, as persuasively and honestly as possible, "
        f"in 3-4 sentences.\n\nTopic: {query}"
    ).content
    against_case = llm.invoke(
        f"Argue AGAINST the following, as persuasively and honestly as possible, "
        f"in 3-4 sentences.\n\nTopic: {query}"
    ).content
    console.print("    [for] " + for_case[:80] + "...")
    console.print("    [against] " + against_case[:80] + "...")

    verdict = llm.invoke(
        f"""You are an impartial judge. Weigh the two stances below on the topic and give a
balanced verdict that acknowledges genuine trade-offs rather than declaring one side
simply "right".

Topic: {query}

FOR: {for_case}

AGAINST: {against_case}

Verdict:"""
    )
    return verdict.content


## Phase 2: The Meta-Controller

**What we are going to do:**
Build the Meta-Controller itself: a structured-output classifier that looks at the incoming task and picks one of the four strategies above, plus a registry mapping each strategy name to its callable, plus a single `meta_controller` entry point that classifies then dispatches.

In [ ]:
# ============ META-CONTROLLER SCHEMA ============
class Strategy(str, Enum):
    DIRECT_ANSWER = "direct_answer"
    REACT_TOOL_USE = "react_tool_use"
    MULTI_STEP_PLANNING = "multi_step_planning"
    DEBATE = "debate"


class MetaControllerDecision(BaseModel):
    """The Meta-Controller's choice of which whole architecture should handle the task."""
    strategy: Strategy = Field(description="The single best-suited strategy for this task.")
    reasoning: str = Field(
        description="One to two sentences explaining why this strategy fits the task's shape."
    )


STRATEGY_DESCRIPTIONS = """\
- direct_answer: The task is a simple, self-contained factual question the model already
  knows (no live/external data needed, no decomposition needed, not a matter of opinion).
- react_tool_use: The task requires current, live, or external information (news, prices,
  recent events, anything that could have changed since training) that a search tool can find.
- multi_step_planning: The task bundles multiple distinct sub-questions or steps that each
  need to be answered and then combined into one coherent final answer.
- debate: The task asks for an opinion, recommendation, or judgment about a trade-off with
  genuine merit on more than one side (no single factual answer exists).
"""

STRATEGY_REGISTRY = {
    Strategy.DIRECT_ANSWER: run_direct_answer,
    Strategy.REACT_TOOL_USE: run_react_tool_use,
    Strategy.MULTI_STEP_PLANNING: run_multi_step_planning,
    Strategy.DEBATE: run_debate,
}


In [ ]:
# ============ META-CONTROLLER NODE ============
def meta_controller(query: str) -> dict:
    """Classify which agentic architecture best fits the task, then dispatch to it."""
    console.print(f"\n=== META-CONTROLLER: Classifying task ===\n'{query}'")

    classifier_llm = llm.with_structured_output(MetaControllerDecision)
    decision = classifier_llm.invoke(
        f"""You are a Meta-Controller. Your job is to choose which ONE agentic architecture
should handle the task below -- not to answer the task yourself.

Available strategies:
{STRATEGY_DESCRIPTIONS}

Task: {query}

Choose the single best-suited strategy and explain your reasoning."""
    )
    console.print(
        f"--- DECISION: strategy='{decision.strategy.value}' | reasoning: {decision.reasoning} ---"
    )

    strategy_fn = STRATEGY_REGISTRY[decision.strategy]
    output = strategy_fn(query)

    return {
        "query": query,
        "strategy": decision.strategy.value,
        "reasoning": decision.reasoning,
        "output": output,
    }


## Phase 3: Demonstration on Four Contrasting Queries

**What we are going to do:**
Run the Meta-Controller on four queries, each deliberately shaped to fit a *different* one of the four strategies:

1. A simple factual question -> should route to `direct_answer`.
2. A question needing live/current information -> should route to `react_tool_use`.
3. A bundled multi-part question -> should route to `multi_step_planning`.
4. An opinion / trade-off question -> should route to `debate`.

The point isn't that these four always route perfectly (classification can be wrong, per the Weaknesses above) — it's to see the *same controller*, using the *same classification prompt*, pick four genuinely different architectures for four genuinely different task shapes.

In [ ]:
# ============ QUERY 1: SIMPLE FACT -> expect direct_answer ============
result_1 = meta_controller("What is the chemical symbol for gold?")
console.print("\n[bold]Final Output:[/bold]")
console.print(Markdown(result_1["output"]))


**Discussion of the Output:**
This is a self-contained factual lookup with a single unambiguous answer that doesn't depend on anything happening after the model's training cutoff. The Meta-Controller should classify this as `direct_answer` and reasoning should reference exactly that: no external data needed, no decomposition needed, not a matter of opinion. Running the full ReAct loop or a debate on this question would waste calls without improving the answer at all.

In [ ]:
# ============ QUERY 2: NEEDS LIVE INFO -> expect react_tool_use ============
result_2 = meta_controller("What is the current price of Bitcoin in USD today?")
console.print("\n[bold]Final Output:[/bold]")
console.print(Markdown(result_2["output"]))


**Discussion of the Output:**
A live, constantly-changing value is exactly what the model cannot know from training data alone. The Meta-Controller should classify this as `react_tool_use`, and the strategy's own internal loop then decides for itself whether and how many times to call `web_search` before answering — that internal decision is a separate, lower-level concern from the Meta-Controller's architecture-level choice.

In [ ]:
# ============ QUERY 3: MULTI-PART -> expect multi_step_planning ============
result_3 = meta_controller(
    "Name the largest planet in the solar system, explain why it is that large, "
    "and compare its size to Earth."
)
console.print("\n[bold]Final Output:[/bold]")
console.print(Markdown(result_3["output"]))


**Discussion of the Output:**
This task bundles three distinct sub-questions ("which planet", "why is it that large", "how does it compare to Earth") that each deserve their own focused answer before being combined. The Meta-Controller should classify this as `multi_step_planning`, letting the planner decompose it into sub-steps and the synthesizer weave the sub-answers into one coherent response — a single direct-answer call would risk shortchanging one of the three parts.

In [ ]:
# ============ QUERY 4: OPINION / TRADE-OFF -> expect debate ============
result_4 = meta_controller(
    "Should companies adopt a fully remote work policy instead of requiring office attendance?"
)
console.print("\n[bold]Final Output:[/bold]")
console.print(Markdown(result_4["output"]))


**Discussion of the Output:**
There is no single factual answer here — genuine, well-argued positions exist on both sides (flexibility and talent-pool access vs. collaboration and culture). The Meta-Controller should classify this as `debate`, producing a for-stance, an against-stance, and a judged synthesis that acknowledges the real trade-offs, rather than a single-shot LLM guess presenting one side as simply correct.

In [ ]:
# ============ SUMMARY TABLE OF ALL FOUR ROUTING DECISIONS ============
for r in (result_1, result_2, result_3, result_4):
    console.print(f"- '{r['query'][:60]}...' -> [bold]{r['strategy']}[/bold]")


## Summary

**Key takeaways:**
* A Meta-Controller's decision variable is *which entire architecture to run*, not which specialist, tool, or prompt variant to use within one already-chosen architecture — that is what separates it from ordinary routing and from a Blackboard's Controller.
* Each strategy here (`direct_answer`, `react_tool_use`, `multi_step_planning`, `debate`) has its own independent internal control flow — some loop, some call tools, some involve multiple LLM personas — and the Meta-Controller doesn't need to know any of those internals, only *when* each is the right fit.
* Structured output (`MetaControllerDecision` with a `reasoning` field) keeps every routing decision inspectable: you can always ask "why did the controller pick this architecture for this task?" instead of trusting a black-box choice.
* This buys right-sized cost (simple questions stay cheap) at the price of a single point of failure: a misclassified task gets no fallback in this minimal design, and a task that is genuinely a *mixture* of two strategies (e.g. "look up X, then debate whether it's a good idea") isn't handled gracefully by a one-shot, single-strategy choice.
* Adding a fifth strategy later only requires one new callable and one new bullet in `STRATEGY_DESCRIPTIONS` — the controller's classification logic itself doesn't change.